In [ ]:
!pip install -U langchain langchain-core

import langchain.agents as agents

print("Available Agent Components:")
print([name for name in dir(agents) if "agent" in name.lower()])

In [ ]:
def spotify_agent(user_input):
    if "trending" in user_input.lower():
        return "Fetching trending songs from Spotify."
    return "I can help you search for songs on Spotify."

user_input = input("You: ")
print("Agent:", spotify_agent(user_input))

In [ ]:
%%writefile spotify_agent.py
memory = []

def spotify_agent(user_input):
    global memory

    if "what did i ask before" in user_input.lower():
        if memory:
            return "Your last queries were:\n" + "\n".join(
                f"{i}. {query}" for i, query in enumerate(memory, 1)
            )
        return "You have not asked anything yet."

    memory.append(user_input)

    if len(memory) > 2:
        memory.pop(0)

    if "trending" in user_input.lower():
        return "Fetching trending songs from Spotify."

    return "I can help you search for songs on Spotify."


while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        break

    print("Agent:", spotify_agent(user_input))

In [ ]:
%%writefile pdf_agent.py
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

documents = SimpleDirectoryReader(
    input_files=["ipl_match_summary.pdf"]
).load_data()

index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

question = input("Ask a question about the match: ")
response = query_engine.query(question)

print("\nAnswer:", response)

In [ ]:
import os
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

@tool
def calculator(expression: str) -> str:
    """Calculate a mathematical expression."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except:
        return "Invalid mathematical expression"

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

agent = create_agent(
    model=llm,
    tools=[calculator]
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is 15 times 8?"}
    ]
})

print(result["messages"][-1].content)